In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
from pathlib import Path
import pandas as pd

# Find the Kaggle competition dataset
input_dir = Path("/kaggle/input")

train_candidates = list(input_dir.rglob("train.csv"))
test_candidates = list(input_dir.rglob("test.csv"))

print("Train files found:", len(train_candidates))
print("Test files found:", len(test_candidates))

# Load the files only when both are available
if not train_candidates or not test_candidates:
    print("ERROR: Dataset files were not found.")
    print("Please check Add Input in the right sidebar.")

else:
    train_path = train_candidates[0]
    test_path = test_candidates[0]

    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    print("\nDataset loaded successfully!")
    print("Train shape:", train.shape)
    print("Test shape:", test.shape)

    display(train.head())

Train files found: 1
Test files found: 1

Dataset loaded successfully!
Train shape: (1460, 81)
Test shape: (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
def add_features(df):
    df = df.copy()

    # Total area of the house
    df["TotalSF"] = (
        df["TotalBsmtSF"].fillna(0)
        + df["1stFlrSF"].fillna(0)
        + df["2ndFlrSF"].fillna(0)
    )

    # Total number of bathrooms
    df["TotalBathrooms"] = (
        df["FullBath"].fillna(0)
        + 0.5 * df["HalfBath"].fillna(0)
        + df["BsmtFullBath"].fillna(0)
        + 0.5 * df["BsmtHalfBath"].fillna(0)
    )

    # Total porch and deck area
    porch_columns = [
        "OpenPorchSF",
        "3SsnPorch",
        "EnclosedPorch",
        "ScreenPorch",
        "WoodDeckSF"
    ]

    df["TotalPorchSF"] = (
        df[porch_columns].fillna(0).sum(axis=1)
    )

    # House age when the property was sold
    df["HouseAge"] = (
        df["YrSold"] - df["YearBuilt"]
    )

    # Years since the last remodeling
    df["RemodAge"] = (
        df["YrSold"] - df["YearRemodAdd"]
    )

    # Interaction between overall quality and living area
    df["OverallQual_SF"] = (
        df["OverallQual"].fillna(0)
        * df["GrLivArea"].fillna(0)
    )

    return df


# Apply the same feature engineering to both datasets
train = add_features(train)
test = add_features(test)

print("Feature engineering completed successfully!")
print("Train shape after feature engineering:", train.shape)
print("Test shape after feature engineering:", test.shape)

display(
    train[
        [
            "TotalSF",
            "TotalBathrooms",
            "TotalPorchSF",
            "HouseAge",
            "RemodAge",
            "OverallQual_SF",
            "SalePrice"
        ]
    ].head()
)

Feature engineering completed successfully!
Train shape after feature engineering: (1460, 87)
Test shape after feature engineering: (1459, 86)


,TotalSF,TotalBathrooms,TotalPorchSF,HouseAge,RemodAge,OverallQual_SF,SalePrice
0,2566,3.5,61,5,5,11970,208500
1,2524,2.5,298,31,31,7572,181500
2,2706,3.5,42,7,6,12502,223500
3,2473,2.0,307,91,36,12019,140000
4,3343,3.5,276,8,8,17584,250000


In [4]:
# Step 13: Separate features and target

X = train.drop(columns=["SalePrice", "Id"], errors="ignore")
y = np.log1p(train["SalePrice"])

# Split into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training features:", X_train.shape)
print("Validation features:", X_valid.shape)
print("Training target:", y_train.shape)
print("Validation target:", y_valid.shape)

Training features: (1168, 85)
Validation features: (292, 85)
Training target: (1168,)
Validation target: (292,)


In [5]:
# Step 14: Identify numerical and categorical columns

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Number of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

Number of numerical features: 42
Number of categorical features: 43


In [6]:
# Step 14B: Create preprocessing pipelines

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Missing"
            )
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [7]:
# Step 15: Train the baseline Ridge Regression model

baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=10))
    ]
)

# Train the model
baseline_model.fit(X_train, y_train)

# Predict on validation data
valid_pred = baseline_model.predict(X_valid)

# Calculate validation RMSE
rmse = np.sqrt(
    mean_squared_error(y_valid, valid_pred)
)

print("Baseline Model: Ridge Regression")
print("Validation Log-RMSE:", round(rmse, 5))

Baseline Model: Ridge Regression
Validation Log-RMSE: 0.1347
